# LeetCode 1802 — Maximum Value at a Given Index in a Bounded Array bằng Python

Link đề: https://leetcode.com/problems/maximum-value-at-a-given-index-in-a-bounded-array/ · Độ khó: **Medium** · Chủ đề: Binary Search, Greedy, Math

Skeleton LeetCode:

```python
class Solution:
    def maxValue(self, n: int, index: int, maxSum: int) -> int:
        ...
```

Lộ trình: **vét cạn mọi mảng** (chỉ cho `n`, `maxSum` tí hon) → **thử tăng dần đỉnh, dựng mảng tốn ít nhất** `O(ans · n)` → **công thức tổng cấp số cộng `O(1)` + tìm kiếm tuyến tính** → **tìm kiếm nhị phân trên đáp án** `O(log maxSum)` (lời giải nộp) → **đắp từng tầng** `O(min(n, √maxSum))`.

## 1. Mục tiêu học

1. Nhận ra cấu trúc tối ưu: khi đỉnh `nums[index] = v`, mảng **tốn ít nhất** là "kim tự tháp" giảm 1 mỗi bước ra hai bên, chạm đáy 1 thì đi ngang.
2. Tính tổng một phía bằng **cấp số cộng** có **sàn bằng 1**, trong `O(1)`.
3. Áp dụng **tìm kiếm nhị phân trên đáp án** cho hàm **đơn điệu** `cost(v)`; viết đúng mẫu "tìm `v` lớn nhất thỏa điều kiện" (làm tròn `mid` lên).
4. So sánh với cách "đắp tầng" tham lam và hiểu khi nào mỗi cách hợp lý.
5. Kiểm chứng công thức bằng vét cạn ngẫu nhiên có seed; tránh lỗi đếm đỉnh hai lần và quên sàn 1.

## 2. Đề bài & ví dụ

Cho ba số nguyên dương `n`, `index`, `maxSum`. Dựng mảng `nums` (đánh số từ 0) thỏa:
- `nums.length == n`;
- mọi `nums[i]` là **số nguyên dương**;
- `abs(nums[i] - nums[i+1]) ≤ 1` với mọi `0 ≤ i < n-1`;
- tổng mọi phần tử **không vượt quá** `maxSum`;
- `nums[index]` **lớn nhất có thể**.

Trả về `nums[index]` của mảng đó.

**Ví dụ 1:** `n = 4, index = 2, maxSum = 6` → `2`. Một mảng: `[1,2,2,1]`. Không có mảng hợp lệ nào với `nums[2] = 3`.

**Ví dụ 2:** `n = 6, index = 1, maxSum = 10` → `3` (ví dụ `[2,3,2,1,1,1]`, tổng 10).

**Ràng buộc:** `1 ≤ n ≤ maxSum ≤ 10⁹`, `0 ≤ index < n`.

In [1]:
import random
import time
from bisect import bisect_right
from functools import lru_cache
from typing import List

examples = [((4, 2, 6), 2), ((6, 1, 10), 3)]
for (n, index, maxSum), want in examples:
    print(f"n={n}, index={index}, maxSum={maxSum} -> kỳ vọng {want}")

n=4, index=2, maxSum=6 -> kỳ vọng 2
n=6, index=1, maxSum=10 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1 Tổng cấp số cộng

Tổng các số nguyên liên tiếp từ `a` tới `b` (`a ≤ b`) là `(a + b)(b − a + 1) / 2`. Đặc biệt `1 + 2 + … + x = x(x+1)/2`.

In [2]:
a, b = 3, 7
print(sum(range(a, b + 1)), (a + b) * (b - a + 1) // 2)
x = 10
print(sum(range(1, x + 1)), x * (x + 1) // 2)

25 25
55 55


### 3.2 Dãy giảm có sàn

Một phía của "kim tự tháp" có `cnt` phần tử, bắt đầu từ `x` (sát đỉnh) và giảm 1 mỗi bước, nhưng **không xuống dưới 1**:
- *Nếu* `x ≥ cnt`: các giá trị là `x, x−1, …, x−cnt+1` (đều ≥ 1).
- *Nếu* `x < cnt`: `x, x−1, …, 1` rồi `cnt − x` số 1.

In [3]:
def side_list(x, cnt):
    return [max(x - k, 1) for k in range(cnt)]

print(side_list(5, 3), side_list(2, 5), side_list(0, 2))

[5, 4, 3] [2, 1, 1, 1, 1] [1, 1]


### 3.3 Tìm kiếm nhị phân "giá trị lớn nhất thỏa điều kiện"

Nếu `ok(v)` đúng với mọi `v ≤ v*` và sai với mọi `v > v*` (đơn điệu), tìm `v*` bằng:

```python
lo, hi = thấp_nhất_chắc_đúng, cao_nhất_có_thể
while lo < hi:
    mid = (lo + hi + 1) // 2     # làm tròn LÊN
    if ok(mid): lo = mid
    else:       hi = mid - 1
```

Làm tròn lên để tránh vòng lặp vô hạn khi `hi = lo + 1` (xem Lỗi 2).

In [4]:
def last_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid): lo = mid
        else: hi = mid - 1
    return lo

print("căn bậc hai nguyên của 1000:", last_true(0, 1000, lambda v: v * v <= 1000))

căn bậc hai nguyên của 1000: 31


### 3.4 `bisect_right` với tham số `key` (Python ≥ 3.10)

`bisect_right(range(lo, hi+1), target, key=f)` tìm vị trí chèn `target` trong dãy **tăng** `f(lo), f(lo+1), …` — tức số phần tử có `f(v) ≤ target`. `range` không tạo list thật nên vẫn `O(log)`.

In [5]:
cnt = bisect_right(range(1, 1001), 1000, key=lambda v: v * v)
print("số v trong 1..1000 có v² ≤ 1000:", cnt)

số v trong 1..1000 có v² ≤ 1000: 31


### 3.5 Số nguyên lớn

`maxSum` tới `10⁹` nên tổng trung gian tới cỡ `v²` ≈ `10¹⁸` — Python không tràn; C++/Java phải dùng 64-bit.

In [6]:
v = 10**9
print(v * (v + 1) // 2, (v * (v + 1) // 2) > 2**63 - 1)

500000000500000000 False


## 4. Ý tưởng nền tảng / trực giác

Cố định đỉnh `nums[index] = v`. Để **tổng nhỏ nhất**, mỗi phần tử nên nhỏ nhất có thể: phần tử cách đỉnh `d` bước phải `≥ v − d` (vì mỗi bước giảm tối đa 1) và `≥ 1`. Chọn đúng `max(v − d, 1)` là hợp lệ ⇒ mảng "kim tự tháp có sàn".

Ví dụ 2 (`n = 6, index = 1`), thử `v = 3`:

```
chỉ số:   0  1  2  3  4  5
mảng:     2  3  2  1  1  1     tổng = 10 ≤ 10 ✔
```

Thử `v = 4`: `[3,4,3,2,1,1]` tổng 14 > 10 ✘. Vậy đáp án 3.

Đặt `cost(v)` = tổng nhỏ nhất khi đỉnh là `v`. `cost` **tăng ngặt** theo `v` (tăng đỉnh thì không phần tử nào giảm) ⇒ đáp án là `v` lớn nhất với `cost(v) ≤ maxSum`.

In [7]:
def min_array(n, index, v):
    return [max(v - abs(i - index), 1) for i in range(n)]

for v in [1, 2, 3, 4]:
    arr = min_array(6, 1, v)
    print(f"v={v}: {arr} tổng={sum(arr)}")

v=1: [1, 1, 1, 1, 1, 1] tổng=6
v=2: [1, 2, 1, 1, 1, 1] tổng=7
v=3: [2, 3, 2, 1, 1, 1] tổng=10
v=4: [3, 4, 3, 2, 1, 1] tổng=14


## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn mọi mảng hợp lệ (chuẩn tuyệt đối, chỉ cho input tí hon)

DFS dựng từng phần tử, mỗi phần tử trong `[1, maxSum]`, ràng buộc chênh lệch ≤ 1 và tổng ≤ `maxSum`; lưu `nums[index]` lớn nhất. Không dùng nhận xét "kim tự tháp" nào ⇒ dùng để kiểm chứng chính nhận xét đó.

In [8]:
def max_value_enumerate(n, index, maxSum):
    best = 0
    arr = []
    def dfs(i, total):
        nonlocal best
        if i == n:
            best = max(best, arr[index])
            return
        cands = range(1, maxSum + 1) if i == 0 else range(max(1, arr[-1] - 1), arr[-1] + 2)
        for v in cands:
            if total + v + (n - i - 1) > maxSum:     # các phần tử còn lại ít nhất là 1
                break
            arr.append(v)
            dfs(i + 1, total + v)
            arr.pop()
    dfs(0, 0)
    return best

for (n, index, maxSum), want in examples:
    print(max_value_enumerate(n, index, maxSum), want)

2 2
3 3


### 5.2 Baseline thực dụng: tăng dần `v`, dựng mảng tốn ít nhất

Dùng nhận xét ở mục 4: thử `v = 1, 2, …`, tính `sum(min_array(n, index, v))`; dừng khi vượt `maxSum`.

In [9]:
def max_value_linear_build(n, index, maxSum):
    v = 1
    while sum(max(v + 1 - abs(i - index), 1) for i in range(n)) <= maxSum:
        v += 1
    return v

for (n, index, maxSum), want in examples:
    print(max_value_linear_build(n, index, maxSum), want)

2 2
3 3


**Độ phức tạp:** đáp án có thể tới `~maxSum` (khi `n = 1`), mỗi lần thử `O(n)` ⇒ `O(ans · n)`. Với `n = 1, maxSum = 10⁹` là `10⁹` vòng; với `n = 10⁹` mỗi vòng là `10⁹` phép. Hai nút thắt: (1) tính `cost(v)` mất `O(n)` — thay bằng **công thức `O(1)`**; (2) thử **từng** `v` — thay bằng **tìm kiếm nhị phân** nhờ tính đơn điệu.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — công thức `O(1)`:** với `side(x, cnt)` là tổng một phía (3.2):
- *Nếu* `x ≥ cnt` *thì* `side = (x + (x − cnt + 1)) · cnt / 2`.
- *Nếu* `x < cnt` *thì* `side = x(x+1)/2 + (cnt − x)`.

`cost(v) = v + side(v − 1, index) + side(v − 1, n − 1 − index)` — bên trái có `index` phần tử, bên phải có `n − 1 − index` phần tử, **đỉnh chỉ tính một lần**.

**Insight 2 — đơn điệu:** `cost(v+1) > cost(v)` ⇒ tập `{v : cost(v) ≤ maxSum}` là một **tiền tố** `1..v*`. Ràng buộc `n ≤ maxSum` bảo đảm `cost(1) = n ≤ maxSum` nên `v* ≥ 1`.

**Insight 3 — cận trên:** `cost(v) ≥ v + (n − 1)` ⇒ `v ≤ maxSum − n + 1`. Tìm kiếm nhị phân trên `[1, maxSum − n + 1]` (hoặc đơn giản `[1, maxSum]`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Công thức `O(1)` + tìm kiếm tuyến tính

Bước đệm: chỉ thay `O(n)` bằng `O(1)`.

In [10]:
def side(x, cnt):
    """Tổng cnt phần tử: x, x-1, ..., không nhỏ hơn 1."""
    if cnt <= 0:
        return 0
    if x >= cnt:
        return (x + x - cnt + 1) * cnt // 2
    return x * (x + 1) // 2 + (cnt - x)

def cost(n, index, v):
    return v + side(v - 1, index) + side(v - 1, n - 1 - index)

for v in range(1, 6):
    assert cost(6, 1, v) == sum(min_array(6, 1, v))
    print(f"cost(v={v}) = {cost(6, 1, v)}")

def max_value_linear_formula(n, index, maxSum):
    v = 1
    while cost(n, index, v + 1) <= maxSum:
        v += 1
    return v

for (n, index, maxSum), want in examples:
    print(max_value_linear_formula(n, index, maxSum), want)

cost(v=1) = 6
cost(v=2) = 7
cost(v=3) = 10
cost(v=4) = 14
cost(v=5) = 19
2 2
3 3


**Độ phức tạp:** `O(ans)` — vẫn có thể `10⁹` vòng khi `n` nhỏ.

### 7.2 Cách 3 — Tìm kiếm nhị phân trên đáp án (`O(log maxSum)`) — lời giải chuẩn

In [11]:
def max_value_binary_demo(n, index, maxSum):
    lo, hi = 1, maxSum - n + 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        c = cost(n, index, mid)
        print(f"lo={lo:>3} hi={hi:>3} mid={mid:>3} cost={c:>4} {'<=' if c <= maxSum else '> '} {maxSum}")
        if c <= maxSum:
            lo = mid
        else:
            hi = mid - 1
    return lo

print("=>", max_value_binary_demo(6, 1, 10))

lo=  1 hi=  5 mid=  3 cost=  10 <= 10
lo=  3 hi=  5 mid=  4 cost=  14 >  10
=> 3


In [12]:
class SolutionBinary:
    def maxValue(self, n: int, index: int, maxSum: int) -> int:
        lo, hi = 1, maxSum - n + 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if cost(n, index, mid) <= maxSum:
                lo = mid
            else:
                hi = mid - 1
        return lo

class SolutionBisect:
    def maxValue(self, n: int, index: int, maxSum: int) -> int:
        # số v trong [1, maxSum] có cost(v) <= maxSum  ==  v lớn nhất thỏa
        return bisect_right(range(1, maxSum + 1), maxSum, key=lambda v: cost(n, index, v))

for (n, index, maxSum), want in examples:
    print(SolutionBinary().maxValue(n, index, maxSum), SolutionBisect().maxValue(n, index, maxSum), want)

2 2 2
3 3 3


**Độ phức tạp:** `O(log maxSum)` lần gọi `cost` `O(1)` ⇒ `O(log maxSum)` thời gian (~30 vòng), `O(1)` bộ nhớ.

### 7.3 Cách 4 — Đắp từng tầng (tham lam)

Bắt đầu mảng toàn 1 (tốn `n`, còn dư `rest = maxSum − n`). Mỗi lần nâng đỉnh thêm 1, ta phải nâng **cả một tầng**: ở tầng thứ `k` (tính từ 0), đoạn bị nâng là `[index − k, index + k]` cắt theo biên mảng, độ rộng `w_k = min(k, index) + min(k, n−1−index) + 1`. Khi `w_k = n` (phủ cả mảng), mọi tầng sau đều tốn `n` ⇒ chia một lần.

In [13]:
class SolutionLayers:
    def maxValue(self, n: int, index: int, maxSum: int) -> int:
        rest = maxSum - n          # mảng toàn 1
        v = 1
        k = 0
        while True:
            w = min(k, index) + min(k, n - 1 - index) + 1
            if w == n:             # tầng đã phủ hết: các tầng sau đều tốn n
                return v + rest // n
            if rest < w:
                return v
            rest -= w
            v += 1
            k += 1

for (n, index, maxSum), want in examples:
    print(SolutionLayers().maxValue(n, index, maxSum), want)

2 2
3 3


**Độ phức tạp:** số tầng trước khi phủ hết ≤ `n`. Hơn nữa, khi tầng `k` chưa phủ hết thì ít nhất một phía chưa chạm biên, phía đó góp đủ `k` ô ⇒ `w_k ≥ k + 1`. Tổng chi phí `K` tầng đầu ≥ `K(K+1)/2 ≤ maxSum` ⇒ `K ≤ √(2·maxSum) ≈ 45000`. Vậy `O(min(n, √maxSum))` — chậm hơn nhị phân nhưng trực quan, không cần công thức.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 1 (`n = 4, index = 2, maxSum = 6`): bên trái 2 phần tử, bên phải 1 phần tử, `hi = 6 − 4 + 1 = 3`.

| `lo` | `hi` | `mid` | mảng tốn ít nhất | `cost(mid)` | quyết định |
|---|---|---|---|---|---|
| 1 | 3 | 2 | `[1,1,2,1]` | 5 ≤ 6 | `lo = 2` |
| 2 | 3 | 3 | `[1,2,3,2]` | 8 > 6 | `hi = 2` |
| 2 | 2 | — | — | — | dừng, đáp án **2** |

(Mảng `[1,2,2,1]` trong đề cũng hợp lệ với tổng 6; mảng tốn ít nhất có tổng 5.)

In [14]:
def trace_binary(n, index, maxSum):
    lo, hi = 1, maxSum - n + 1
    step = 0
    while lo < hi:
        step += 1
        mid = (lo + hi + 1) // 2
        L, R = side(mid - 1, index), side(mid - 1, n - 1 - index)
        c = mid + L + R
        print(f"vòng {step}: lo={lo}, hi={hi}, mid={mid} | trái={L} + đỉnh={mid} + phải={R} = {c} "
              f"| mảng={min_array(n, index, mid) if n <= 12 else '...'} -> {'lo=mid' if c <= maxSum else 'hi=mid-1'}")
        if c <= maxSum: lo = mid
        else: hi = mid - 1
    print("đáp án:", lo)
    return lo

trace_binary(4, 2, 6); print()
trace_binary(6, 1, 10)

vòng 1: lo=1, hi=3, mid=2 | trái=2 + đỉnh=2 + phải=1 = 5 | mảng=[1, 1, 2, 1] -> lo=mid
vòng 2: lo=2, hi=3, mid=3 | trái=3 + đỉnh=3 + phải=2 = 8 | mảng=[1, 2, 3, 2] -> hi=mid-1
đáp án: 2

vòng 1: lo=1, hi=5, mid=3 | trái=2 + đỉnh=3 + phải=5 = 10 | mảng=[2, 3, 2, 1, 1, 1] -> lo=mid
vòng 2: lo=3, hi=5, mid=4 | trái=3 + đỉnh=4 + phải=7 = 14 | mảng=[3, 4, 3, 2, 1, 1] -> hi=mid-1
đáp án: 3


3

Trace Cách 4 (đắp tầng) trên Ví dụ 2:

In [15]:
def trace_layers(n, index, maxSum):
    rest, v, k = maxSum - n, 1, 0
    print(f"mảng toàn 1, dư {rest}")
    while True:
        w = min(k, index) + min(k, n - 1 - index) + 1
        if w == n:
            print(f"tầng {k} phủ hết (w={n}): thêm {rest // n} tầng -> v={v + rest // n}")
            return v + rest // n
        if rest < w:
            print(f"tầng {k} cần {w} > dư {rest}: dừng, v={v}")
            return v
        rest -= w; v += 1
        print(f"tầng {k}: nâng đoạn [{max(0, index - k)}, {min(n - 1, index + k)}] rộng {w} -> v={v}, dư {rest}")
        k += 1

trace_layers(6, 1, 10)

mảng toàn 1, dư 4
tầng 0: nâng đoạn [1, 1] rộng 1 -> v=2, dư 3
tầng 1: nâng đoạn [0, 2] rộng 3 -> v=3, dư 0
tầng 2 cần 4 > dư 0: dừng, v=3


3

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def maxValue(self, n: int, index: int, maxSum: int) -> int:
        def side(x: int, cnt: int) -> int:
            # tổng cnt số: x, x-1, ..., không xuống dưới 1
            if x >= cnt:
                return (2 * x - cnt + 1) * cnt // 2
            return x * (x + 1) // 2 + (cnt - x)

        def cost(v: int) -> int:
            return v + side(v - 1, index) + side(v - 1, n - 1 - index)

        lo, hi = 1, maxSum - n + 1
        while lo < hi:
            mid = (lo + hi + 1) // 2        # tìm v LỚN NHẤT thỏa -> làm tròn lên
            if cost(mid) <= maxSum:
                lo = mid
            else:
                hi = mid - 1
        return lo

for (n, index, maxSum), want in examples:
    assert Solution().maxValue(n, index, maxSum) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (cận dưới từng phần tử):** nếu `nums[index] = v` thì `nums[i] ≥ max(v − |i − index|, 1)`. *Chứng minh:* đi từ `index` tới `i` mất `|i − index|` bước, mỗi bước giảm tối đa 1; và mọi phần tử ≥ 1.

**Bổ đề 2 (đạt được):** mảng `m_i = max(v − |i − index|, 1)` thỏa mọi điều kiện (trừ tổng): hai phần tử kề nhau có `|i − index|` chênh 1 nên `v − |…|` chênh 1; sau khi lấy `max(·, 1)` chênh lệch vẫn ≤ 1. Vậy `cost(v) = Σ m_i` là **tổng nhỏ nhất** với đỉnh `v`.

**Bổ đề 3 (đơn điệu):** `m_i(v+1) ≥ m_i(v)` với mọi `i`, và `m_index` tăng đúng 1 ⇒ `cost(v+1) > cost(v)`.

**Kết luận:** tồn tại mảng hợp lệ với đỉnh `v` ⇔ `cost(v) ≤ maxSum`. Tập `v` thỏa là `[1, v*]` (khác rỗng vì `cost(1) = n ≤ maxSum`). Tìm kiếm nhị phân với bất biến "`ok(lo)` đúng, mọi `v > hi` sai" trả về `v*`. Công thức `side` đúng theo 3.1–3.2 (đã kiểm bằng `min_array` ở mục 7.1).

## 11. Phân tích Time / Space

Gọi `A` = đáp án (≤ `maxSum − n + 1`).

| Cách | Time | Space |
|---|---|---|
| 1a. Vét cạn mọi mảng | hàm mũ | `O(n)` |
| 1b. Tăng `v`, dựng mảng | `O(A · n)` | `O(1)` |
| 2. Tăng `v`, công thức | `O(A)` | `O(1)` |
| 3. Nhị phân | `O(log maxSum)` | `O(1)` |
| 4. Đắp tầng | `O(min(n, √maxSum))` | `O(1)` |

In [17]:
cases = [(1, 0, 10**9), (10**9, 0, 10**9), (10**5, 50000, 10**9), (3, 1, 10**9), (10**9 // 2, 12345, 10**9)]
for n, index, maxSum in cases:
    res = []
    for name, sol in [("nhị phân", Solution()), ("bisect", SolutionBisect()), ("tầng", SolutionLayers())]:
        t = time.perf_counter()
        r = sol.maxValue(n, index, maxSum)
        res.append(r)
        dt = (time.perf_counter() - t) * 1000
        print(f"n={n}, index={index}, maxSum={maxSum} | {name}: {r} ({dt:.2f} ms)")
    assert len(set(res)) == 1

n=1, index=0, maxSum=1000000000 | nhị phân: 1000000000 (0.01 ms)
n=1, index=0, maxSum=1000000000 | bisect: 1000000000 (0.01 ms)
n=1, index=0, maxSum=1000000000 | tầng: 1000000000 (0.00 ms)
n=1000000000, index=0, maxSum=1000000000 | nhị phân: 1 (0.00 ms)
n=1000000000, index=0, maxSum=1000000000 | bisect: 1 (0.01 ms)
n=1000000000, index=0, maxSum=1000000000 | tầng: 1 (0.00 ms)
n=100000, index=50000, maxSum=1000000000 | nhị phân: 31622 (0.01 ms)
n=100000, index=50000, maxSum=1000000000 | bisect: 31622 (0.01 ms)
n=100000, index=50000, maxSum=1000000000 | tầng: 31622 (3.44 ms)
n=3, index=1, maxSum=1000000000 | nhị phân: 333333334 (0.01 ms)
n=3, index=1, maxSum=1000000000 | bisect: 333333334 (0.01 ms)
n=3, index=1, maxSum=1000000000 | tầng: 333333334 (0.00 ms)
n=500000000, index=12345, maxSum=1000000000 | nhị phân: 23777 (0.01 ms)
n=500000000, index=12345, maxSum=1000000000 | bisect: 23777 (0.01 ms)
n=500000000, index=12345, maxSum=1000000000 | tầng: 23777 (2.77 ms)


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
sols = [Solution(), SolutionBinary(), SolutionBisect(), SolutionLayers()]
edge = examples + [
    ((1, 0, 1), 1),
    ((1, 0, 7), 7),               # một phần tử: lấy hết
    ((3, 1, 3), 1),               # maxSum == n: toàn 1
    ((3, 1, 4), 2),               # [1,2,1]
    ((3, 0, 4), 2),               # [2,1,1]
    ((5, 0, 5), 1),
    ((4, 0, 4), 1),
    ((3, 2, 18), 7),              # [5,6,7] tổng 18
]
for (n, index, maxSum), want in edge:
    for s in sols:
        assert s.maxValue(n, index, maxSum) == want, (n, index, maxSum, s)
    if maxSum <= 20:
        assert max_value_enumerate(n, index, maxSum) == want
    assert max_value_linear_build(n, index, maxSum) == want == max_value_linear_formula(n, index, maxSum)
    print("OK", (n, index, maxSum), "->", want)

OK (4, 2, 6) -> 2
OK (6, 1, 10) -> 3
OK (1, 0, 1) -> 1
OK (1, 0, 7) -> 7
OK (3, 1, 3) -> 1
OK (3, 1, 4) -> 2
OK (3, 0, 4) -> 2
OK (5, 0, 5) -> 1
OK (4, 0, 4) -> 1
OK (3, 2, 18) -> 7


### 12.2 Ngẫu nhiên có seed: đối chiếu vét cạn mọi mảng (`n ≤ 6`, `maxSum ≤ 18`)

In [19]:
rng = random.Random(1802)
for _ in range(400):
    n = rng.randint(1, 6)
    maxSum = rng.randint(n, 18)
    index = rng.randrange(n)
    want = max_value_enumerate(n, index, maxSum)
    for s in sols:
        assert s.maxValue(n, index, maxSum) == want, (n, index, maxSum)
for _ in range(300):
    n = rng.randint(1, 300)
    maxSum = rng.randint(n, 5000)
    index = rng.randrange(n)
    want = max_value_linear_build(n, index, maxSum)
    assert all(s.maxValue(n, index, maxSum) == want for s in sols)
print("400 case vét cạn + 300 case vừa (Cách 1b): khớp")

400 case vét cạn + 300 case vừa (Cách 1b): khớp


### 12.3 Kiểm tra công thức `side` và tính đơn điệu của `cost`

In [20]:
for x in range(0, 30):
    for cnt in range(0, 30):
        assert side(x, cnt) == sum(side_list(x, cnt))
rng = random.Random(3)
for _ in range(200):
    n = rng.randint(1, 50); index = rng.randrange(n)
    vals = [cost(n, index, v) for v in range(1, 60)]
    assert all(vals[i] < vals[i + 1] for i in range(len(vals) - 1))
    assert vals[0] == n
print("side() khớp liệt kê; cost() tăng ngặt, cost(1) = n")

side() khớp liệt kê; cost() tăng ngặt, cost(1) = n


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên sàn 1: dùng cấp số cộng "trần trụi"

In [21]:
def wrong_no_floor(n, index, maxSum):
    def side_bad(x, cnt):
        return (x + x - cnt + 1) * cnt // 2          # SAI: các số có thể <= 0
    def cost_bad(v):
        return v + side_bad(v - 1, index) + side_bad(v - 1, n - 1 - index)
    lo, hi = 1, maxSum
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if cost_bad(mid) <= maxSum: lo = mid
        else: hi = mid - 1
    return lo

for args in [(10, 0, 10), (6, 1, 10)]:
    print(args, "sai:", wrong_no_floor(*args), "| đúng:", Solution().maxValue(*args))

(10, 0, 10) sai: 5 | đúng: 1
(6, 1, 10) sai: 3 | đúng: 3


**Vì sao sai:** với `v = 2`, phía phải 9 phần tử bị tính là `1 + 0 + (−1) + … + (−7) = −27` — tổng âm khiến `cost` bị đánh giá thấp, và hàm `cost_bad` còn không đơn điệu. Đề yêu cầu mọi phần tử **dương**. (Ví dụ 2 `(6, 1, 10)` tình cờ vẫn ra đúng 3 — lỗi chỉ lộ khi một phía **dài hơn** đỉnh, nên test mẫu của đề không bắt được.) **Sửa:** nhánh `x < cnt` thêm `cnt − x` số 1.

### Lỗi 2 — Tìm "lớn nhất thỏa" nhưng `mid` làm tròn xuống

In [22]:
def wrong_mid_floor(n, index, maxSum, max_iter=200):
    lo, hi = 1, maxSum
    it = 0
    while lo < hi:
        it += 1
        if it > max_iter:
            return f"không dừng sau {max_iter} vòng (lo={lo}, hi={hi})"
        mid = (lo + hi) // 2                          # SAI với lo = mid
        if cost(n, index, mid) <= maxSum: lo = mid
        else: hi = mid - 1
    return lo

print("sai:", wrong_mid_floor(4, 2, 6), "| đúng:", Solution().maxValue(4, 2, 6))

sai: không dừng sau 200 vòng (lo=1, hi=2) | đúng: 2


**Vì sao sai:** khi `hi = lo + 1` và `ok(lo)` đúng, `mid = lo` ⇒ gán `lo = mid` không thay đổi gì ⇒ lặp vô hạn (ta đặt giới hạn vòng để demo không treo). **Sửa:** `mid = (lo + hi + 1) // 2` khi nhánh thành công là `lo = mid`.

### Lỗi 3 — Tính đỉnh hai lần

In [23]:
def wrong_double_peak(n, index, maxSum):
    def cost_bad(v):
        return side(v, index + 1) + side(v, n - index)   # SAI: cả hai phía đều chứa đỉnh v
    lo, hi = 1, maxSum
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if cost_bad(mid) <= maxSum: lo = mid
        else: hi = mid - 1
    return lo

print("sai:", wrong_double_peak(6, 1, 10), "| đúng:", Solution().maxValue(6, 1, 10))

sai: 2 | đúng: 3


**Vì sao sai:** `side(v, index + 1)` đã gồm đỉnh, `side(v, n − index)` lại gồm đỉnh lần nữa ⇒ `cost` thừa `v`, đáp án bị thấp. **Sửa:** `v + side(v−1, index) + side(v−1, n−1−index)` hoặc trừ `v` một lần.

### Lỗi 4 — Tìm kiếm tuyến tính trên `v` (đúng nhưng TLE)

In [24]:
t = time.perf_counter()
r = max_value_linear_formula(1, 0, 2 * 10**5)
dt = time.perf_counter() - t
print(f"n=1, maxSum=2·10^5: {r} trong {dt*1000:.0f} ms -> với maxSum=10^9 ước tính ≈ {dt * 5000:.0f} s")
print("nhị phân:", Solution().maxValue(1, 0, 10**9))

n=1, maxSum=2·10^5: 200000 trong 25 ms -> với maxSum=10^9 ước tính ≈ 123 s
nhị phân: 1000000000


**Vì sao sai:** đáp án có thể cỡ `10⁹` (ví dụ `n = 1`), lặp từng giá trị là quá chậm dù mỗi bước `O(1)`. **Sửa:** tìm kiếm nhị phân (đơn điệu).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tối đa hóa X sao cho tổng/chi phí ≤ ngân sách"** + chi phí **đơn điệu** theo X ⇒ **tìm kiếm nhị phân trên đáp án**.
- **"Chênh lệch kề nhau ≤ 1"** + "tối thiểu tổng" ⇒ hình **kim tự tháp/núi** quanh điểm cố định, tổng tính bằng cấp số cộng.
- Luôn nghĩ tới **sàn** (giá trị nhỏ nhất cho phép) khi dùng cấp số cộng.
- Mẫu nhị phân: nhánh `lo = mid` ⇒ `mid` làm tròn **lên**; nhánh `hi = mid` ⇒ làm tròn **xuống**.
- Python ≥ 3.10: `bisect_right(range(...), target, key=f)` là cách viết nhị phân "không lỗi off-by-one".

## 15. Biến thể của bài

1. **Chênh lệch kề nhau ≤ `d`** (thay cho 1): phía một bên giảm `d` mỗi bước, công thức `side` đổi thành tổng cấp số cộng công sai `d` có sàn.
2. **Sàn khác 1** (mọi phần tử ≥ `L`): tịnh tiến — đặt `nums[i] = L − 1 + b[i]` với `b[i] ≥ 1`, giải bài gốc với `maxSum − n(L−1)`.
3. **Trả về chính mảng** đạt đáp án (Bài tập 5).
4. **Mảng vòng** (phần tử đầu và cuối cũng kề nhau): khoảng cách thành `min(|i − index|, n − |i − index|)`.
5. Bài cùng mẫu "nhị phân trên đáp án": LeetCode 875 *Koko Eating Bananas*, 1011 *Capacity To Ship Packages*, 2226 *Maximum Candies Allocated to K Children*.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Vét cạn mảng | DFS mọi mảng | mũ | `O(n)` | kiểm chứng nhận xét "kim tự tháp" |
| 1b. Tăng `v` + dựng mảng | `min_array` | `O(A·n)` | `O(1)` | kiểm chứng công thức |
| 2. Tăng `v` + công thức | `cost(v)` `O(1)` | `O(A)` | `O(1)` | `A` nhỏ |
| 3. Nhị phân | `cost` đơn điệu | `O(log maxSum)` | `O(1)` | **nộp bài** |
| 4. Đắp tầng | nâng từng tầng, chia khi phủ hết | `O(min(n, √maxSum))` | `O(1)` | không muốn nhớ công thức |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `maxValue(1, 0, 7)`, `maxValue(3, 1, 4)`, `maxValue(3, 1, 5)`?

**Bài 2 (tính tay):** Với `n = 6, index = 1`, tính `cost(4)` bằng công thức `side` (ghi rõ hai phía).

**Bài 3 (điền TODO):** Viết `side_d(x, cnt, d)` — tổng `cnt` phần tử bắt đầu từ `x`, mỗi bước giảm `d`, không nhỏ hơn 1 (biến thể 1). Gợi ý kiểm bằng liệt kê.

**Bài 4 (tự viết lại):** Viết lại lời giải nộp bằng **`bisect_right` có `key`** chỉ trong 2–3 dòng thân hàm.

**Bài 5 (mini challenge):** Viết `build_array(n, index, maxSum)` trả về **một mảng hợp lệ** đạt `nums[index]` lớn nhất (tổng ≤ `maxSum`). Kiểm tra mọi điều kiện của đề bằng `assert`.

In [25]:
def side_d(x, cnt, d):
    # TODO
    pass

def max_value_bisect(n, index, maxSum):
    # TODO
    pass

def build_array(n, index, maxSum):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `n = 1` thì lấy hết | `[1,2,1]` tổng 4 | `[2,2,1]`? chỉ số 1 lên 3 cần tổng `2+3+2 = 7` |
| 2 | Trái 1 phần tử, phải 4 phần tử | `side(3, 1) = 3` | `side(3, 4) = 3+2+1+1 = 7`, tổng `4+3+7 = 14` |
| 3 | Số phần tử còn > 1: `t = (x − 1) // d + 1` (khi `x ≥ 1`) | Nếu `t ≥ cnt`: cấp số cộng đủ `cnt` số | Ngược lại: `t` số cấp số cộng + `cnt − t` số 1 |
| 4 | `range(1, maxSum + 1)` | `key=lambda v: cost(v)` | Kết quả `bisect_right` chính là đáp án |
| 5 | Tính `v` bằng lời giải | Dựng `max(v − |i − index|, 1)` | Kiểm tổng ≤ `maxSum`, chênh lệch ≤ 1, dương |

## 19. Đáp án tham khảo

In [26]:
r1 = [Solution().maxValue(*a) for a in [(1, 0, 7), (3, 1, 4), (3, 1, 5)]]
print("Bài 1:", r1); assert r1 == [7, 2, 2]
L, R = side(3, 1), side(3, 4)
print("Bài 2: trái", L, "+ đỉnh 4 + phải", R, "=", 4 + L + R); assert 4 + L + R == cost(6, 1, 4) == 14

Bài 1: [7, 2, 2]
Bài 2: trái 3 + đỉnh 4 + phải 7 = 14


In [27]:
def side_d(x, cnt, d):
    if cnt <= 0:
        return 0
    if x < 1:
        return cnt                                   # mọi phần tử bị kéo lên sàn 1
    t = (x - 1) // d + 1                             # số phần tử x, x-d, ... còn >= 1
    if t >= cnt:
        last = x - (cnt - 1) * d
        return (x + last) * cnt // 2
    last = x - (t - 1) * d
    return (x + last) * t // 2 + (cnt - t)

for x in range(0, 25):
    for cnt in range(0, 15):
        for d in range(1, 5):
            assert side_d(x, cnt, d) == sum(max(x - k * d, 1) for k in range(cnt)), (x, cnt, d)
assert all(side_d(x, c, 1) == side(x, c) for x in range(1, 30) for c in range(0, 30))
print("Bài 3: OK")

def max_value_bisect(n, index, maxSum):
    return bisect_right(range(1, maxSum + 1), maxSum, key=lambda v: cost(n, index, v))

rng = random.Random(10)
for _ in range(300):
    n = rng.randint(1, 10**6); maxSum = rng.randint(n, 10**9); index = rng.randrange(n)
    assert max_value_bisect(n, index, maxSum) == Solution().maxValue(n, index, maxSum)
print("Bài 4: OK")

Bài 3: OK
Bài 4: OK


In [28]:
def build_array(n, index, maxSum):
    v = Solution().maxValue(n, index, maxSum)
    return [max(v - abs(i - index), 1) for i in range(n)]

def check_array(nums, n, index, maxSum):
    assert len(nums) == n
    assert all(x >= 1 for x in nums)
    assert all(abs(nums[i] - nums[i + 1]) <= 1 for i in range(n - 1))
    assert sum(nums) <= maxSum

rng = random.Random(77)
for _ in range(300):
    n = rng.randint(1, 8); maxSum = rng.randint(n, 25); index = rng.randrange(n)
    arr = build_array(n, index, maxSum)
    check_array(arr, n, index, maxSum)
    assert arr[index] == max_value_enumerate(n, index, maxSum)
print("Bài 5: OK", build_array(6, 1, 10), build_array(4, 2, 6))

Bài 5: OK [2, 3, 2, 1, 1, 1] [1, 1, 2, 1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def side(x, cnt):                      # x, x-1, ..., sàn 1, đủ cnt số
    if x >= cnt: return (2*x - cnt + 1) * cnt // 2
    return x * (x + 1) // 2 + (cnt - x)

cost = lambda v: v + side(v-1, index) + side(v-1, n-1-index)
lo, hi = 1, maxSum - n + 1
while lo < hi:
    mid = (lo + hi + 1) // 2           # làm tròn LÊN vì nhánh lo = mid
    if cost(mid) <= maxSum: lo = mid
    else: hi = mid - 1
return lo
```

**Checklist:**
- [ ] `side` có xử lý **sàn 1** (nhánh `x < cnt`)?
- [ ] Đỉnh chỉ tính **một lần**; hai phía dùng `v − 1`?
- [ ] Số phần tử hai phía: `index` và `n − 1 − index`?
- [ ] `mid` làm tròn lên với mẫu `lo = mid`?
- [ ] Đã thử `n = 1`, `maxSum = n`, `index` ở biên (0 và `n − 1`)?